In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import torch.optim
from torch.nn.utils.rnn import pack_padded_sequence
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
df = pd.read_csv(r"C:\Users\arjun\Downloads\charting-m-points-2020s.csv")

In [ ]:
df.info()

In [ ]:
df.isnull().sum()


match_id         0
Pt               0
Set1             0
Set2             0
Gm1              0
Gm2              1
Pts              0
Gm#              0
TbSet            1
Svr              0
1st              0
2nd         345516
Notes       471133
PtWinner         0
dtype: int64

In [ ]:
drop_cols = ['1st','2nd','Notes']
df = df.drop(columns=drop_cols)

In [ ]:
df['Gm2']=df['Gm2'].fillna(0).astype(int)
df['TbSet']=df['TbSet'].fillna(True)    
df.isnull().sum()

match_id    0
Pt          0
Set1        0
Set2        0
Gm1         0
Gm2         0
Pts         0
Gm#         0
TbSet       0
Svr         0
PtWinner    0
dtype: int64

In [ ]:
score = df['Pts'].str.split('-', expand =True)
df['P1_pts'] = score[0]
df['P2_pts'] = score[1]


In [ ]:
df.head()

,match_id,Pt,Set1,Set2,Gm1,Gm2,Pts,Gm#,TbSet,Svr,PtWinner,P1_pts,P2_pts
0,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,92,0,1,1,0,0-15,14,True,2,2,0,15
1,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,93,0,1,1,0,15-15,14,True,2,1,15,15
2,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,94,0,1,1,0,15-30,14,True,2,2,15,30
3,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,95,0,1,1,0,30-30,14,True,2,2,30,30
4,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,96,0,1,1,0,40-30,14,True,2,1,40,30


In [ ]:
def convert(score):
    mapping = {
        "0": 0,
        "15": 1,
        "30": 2,
        "40": 3,
        "AD": 4
    }
    score = str(score)
    if score in mapping:
        return mapping[score]
    return int(score)

In [ ]:
df['P1_pts']=df['P1_pts'].apply(convert)
df['P2_pts']= df['P2_pts'].apply(convert)
df.head()

,match_id,Pt,Set1,Set2,Gm1,Gm2,Pts,Gm#,TbSet,Svr,PtWinner,P1_pts,P2_pts
0,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,92,0,1,1,0,0-15,14,True,2,2,0,1
1,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,93,0,1,1,0,15-15,14,True,2,1,1,1
2,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,94,0,1,1,0,15-30,14,True,2,2,1,2
3,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,95,0,1,1,0,30-30,14,True,2,2,2,2
4,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,96,0,1,1,0,40-30,14,True,2,1,3,2


In [ ]:
df['Svr'] = df['Svr'] -1
df['Svr'].unique()

array([1, 0])

In [ ]:
df['set_diff'] = df['Set1'] - df['Set2']
df['game_diff'] = df['Gm1'] - df['Gm2']
df['point_diff'] = df['P1_pts'] - df['P2_pts']

In [ ]:
match_winner = (df.groupby('match_id')["PtWinner"].last().rename('MatchWinner'))
df = df.merge(match_winner, on='match_id')
df.head()

,match_id,Pt,Set1,Set2,Gm1,Gm2,Pts,Gm#,TbSet,Svr,PtWinner,P1_pts,P2_pts,set_diff,game_diff,point_diff,MatchWinner
0,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,92,0,1,1,0,0-15,14,True,1,2,0,1,-1,1,-1,1
1,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,93,0,1,1,0,15-15,14,True,1,1,1,1,-1,1,0,1
2,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,94,0,1,1,0,15-30,14,True,1,2,1,2,-1,1,-1,1
3,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,95,0,1,1,0,30-30,14,True,1,2,2,2,-1,1,0,1
4,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,96,0,1,1,0,40-30,14,True,1,1,3,2,-1,1,1,1


In [ ]:
feature_columns = [
    "Set1",
    "Set2",
    "Gm1",
    "Gm2",
    "P1_pts",
    "P2_pts",
    "Svr",
    "set_diff",
    "game_diff",
    "point_diff"
]

target_column = "MatchWinner"

In [ ]:
match_ids = df['match_id'].unique()
train_matches, test_matches = train_test_split(match_ids, train_size=0.8)
df_train = df[df['match_id'].isin(train_matches)].copy()
df_test = df[df['match_id'].isin(test_matches)].copy()

In [ ]:
def generate_sequences(df, feature_columns, target_column):

    X = []
    y = []

    for _, match in df.groupby("match_id"):

        features = match[feature_columns].values.astype(np.float32)
        target = match[target_column].iloc[0] - 1      # classes: 0 and 1

        for i in range(1, len(match)+1,5):

            X.append(features[:i])
            y.append(target)

    return X, np.array(y)

In [ ]:
X_train, y_train = generate_sequences(df_train, feature_columns, target_column)
X_test, y_test = generate_sequences(df_test, feature_columns, target_column)


In [ ]:
def pad_feature_sequences(sequences, max_length):

    n_features = sequences[0].shape[1]

    X = np.zeros((len(sequences), max_length, n_features),
                 dtype=np.float32)
    lengths = []
    for i, seq in enumerate(sequences):

        seq = seq[-max_length:]        
        lengths.append(len(seq))
        X[i,:len(seq)] = seq

    return X, np.array(lengths)

In [ ]:
X_train, train_lengths = pad_feature_sequences(X_train, 100)

X_test, test_lengths = pad_feature_sequences(X_test, 100)

In [ ]:

class TennisDataset(Dataset):

    def __init__(self, X, y):

        self.X = X
        self.y = y

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        sequence = torch.tensor(
            self.X[idx],
            dtype=torch.float32
        )

        target = torch.tensor(
            self.y[idx],
            dtype=torch.long
        )

        length = len(sequence)

        return sequence, length, target
    


def collate_fn(batch):

    sequences = [item[0] for item in batch]

    lengths = torch.tensor(
        [item[1] for item in batch],
        dtype=torch.long
    )

    targets = torch.tensor(
        [item[2] for item in batch],
        dtype=torch.long
    )

    padded = pad_sequence(
        sequences,
        batch_first=True
    )

    return padded, lengths, targets

In [ ]:

train_dataset = TennisDataset(
    X_train,
    y_train
)

test_dataset = TennisDataset(
    X_test,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)